In [ ]:
"""
Скрипт за изтегляне на 10×10 км квадрати от Sentinel‑2 L2A за 2024 г.
за всеки пожар от CSV файла, за месеците май, юли и септември.
Ако няма изображение в целевия месец, търси следващия.

Script for downloading 10×10 km Sentinel‑2 L2A squares for 2024
for each fire from the CSV file, for the months May, July and September.
If no image exists in the target month, searches the next month.
"""

import pandas as pd
import pystac_client
import planetary_computer
from odc.stac import stac_load
import yaml
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle
import os
from rasterio.transform import from_bounds, from_origin
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.crs import CRS
from pyproj import Transformer
import math
from datetime import datetime, timedelta
import pytz
from scipy.ndimage import zoom
import warnings
import traceback
import calendar

warnings.filterwarnings('ignore')

# ======================== ПЪТИЩА / PATHS ============================
INPUT_CSV = r'D:\data\master_thesis\input\fires_suggestion.csv'
OUTPUT_DIR = r'D:\data\master_thesis\exports\sentinel2_fire_images\composite_pictures'

# ======================== КОНФИГУРАЦИЯ / CONFIGURATION ====================
SQUARE_SIZE_KM = 10                       # размер на квадрата в км / square size in km
MAX_CLOUD_COVER = 10                      # максимална облачност, % / max cloud cover, %
MAX_NODATA_PERCENTAGE = 5                 # ако % липсващи пиксели надвиши това – композит
ENHANCED_NODATA_THRESHOLD = 200           # рефлектанс под тази стойност е подозрителен
MIN_COMPLEMENTARY_COVERAGE = 20           # мин. покритие на празни пиксели от допълнителна сцена
MAX_COMPLEMENTARY_IMAGES = 3              # макс. брой сцени за композит
COMPLEMENTARY_SEARCH_WINDOW_DAYS = 30     # прозорец за търсене на допълнителни сцени

def get_utm_epsg(lon):
    """
    Връща правилния EPSG код за UTM зоната на България.
    Returns the correct UTM EPSG code for Bulgaria.
    """
    if 18.0 <= lon <= 24.0:
        return 32634   # UTM зона 34N / UTM zone 34N
    elif 24.0 < lon <= 30.0:
        return 32635   # UTM зона 35N / UTM zone 35N
    else:
        zone = int((lon + 180) / 6) + 1
        return 32600 + zone

# ======================== ПОМОЩНИ ФУНКЦИИ / HELPER FUNCTIONS ==================

def add_scale_bar(ax, transform, scale_km=2):
    """
    Добавя мащабна линия върху изображението.
    Draws a scale bar on the image.
    """
    try:
        y_pixels, x_pixels = ax.images[0].get_array().shape[:2]
        pixel_size_x = transform.a
        pixel_size_y = -transform.e
        pixel_size = (abs(pixel_size_x) + abs(pixel_size_y)) / 2
        scale_length_m = scale_km * 1000
        scale_length_pixels = scale_length_m / pixel_size
        x_pos = x_pixels * 0.95 - scale_length_pixels
        y_pos = y_pixels * 0.95
        rect = Rectangle((x_pos, y_pos), scale_length_pixels, y_pixels * 0.01,
                         facecolor='white', edgecolor='black', linewidth=2,
                         transform=ax.transData)
        ax.add_patch(rect)
        ax.text(x_pos + scale_length_pixels / 2, y_pos - y_pixels * 0.02,
                f'{scale_km} km',
                ha='center', va='top', color='white', fontweight='bold', fontsize=10,
                bbox=dict(boxstyle="round,pad=0.3", facecolor='black', alpha=0.7),
                transform=ax.transData)
    except Exception as e:
        print(f"  ⚠ Неуспех при добавяне на мащаб / Could not add scale bar: {e}")

def display_rgb_from_all_bands(image_path, title):
    """
    Показва RGB изображение от записан GeoTIFF файл.
    Displays an RGB preview from a saved GeoTIFF.
    """
    try:
        with rasterio.open(image_path) as src:
            band_count = src.count
            band_descriptions = [src.descriptions[i] if src.descriptions else f"Band_{i+1}"
                                 for i in range(band_count)]
            metadata = src.tags()
            print(f"  📊 Изображението има {band_count} канала / Image has {band_count} bands: {band_descriptions}")

            red_band = green_band = blue_band = None
            for i, desc in enumerate(band_descriptions):
                if desc == 'B04': red_band = i + 1
                elif desc == 'B03': green_band = i + 1
                elif desc == 'B02': blue_band = i + 1

            if red_band and green_band and blue_band:
                red = src.read(red_band)
                green = src.read(green_band)
                blue = src.read(blue_band)

                total_pixels = red.size
                nodata_pixels = np.sum((red == 0) & (green == 0) & (blue == 0))
                nodata_percentage = (nodata_pixels / total_pixels) * 100
                print(f"  📊 Nodata/черни пиксели / black pixels: {nodata_percentage:.1f}%")

                def stretch_band(band, clip_percentiles=(2, 98)):
                    band_masked = band[band > 0]
                    if len(band_masked) > 0:
                        p_low, p_high = np.percentile(band_masked, clip_percentiles)
                        if p_high > p_low:
                            return np.clip((band - p_low) / (p_high - p_low), 0, 1)
                    return band.astype(np.float32) / band.max() if band.max() > 0 else band.astype(np.float32)

                rgb_stretched = np.dstack([
                    stretch_band(red), stretch_band(green), stretch_band(blue)
                ])
                rgb_stretched = np.power(rgb_stretched, 0.8)

                fig, ax = plt.subplots(figsize=(14, 12))
                ax.imshow(rgb_stretched)
                add_scale_bar(ax, src.transform, scale_km=2)

                full_title = f"{title}\n"
                full_title += (f"Пожар ID / Fire ID: {metadata.get('fire_id','?')} | "
                               f"Месец / Month: {metadata.get('month','?')} | "
                               f"Изображение / Image: {metadata.get('acquisition_date','?')} | "
                               f"Облачност / Cloud: {metadata.get('cloud_cover','?')}% | "
                               f"Черни / Black: {nodata_percentage:.1f}%")
                if metadata.get('is_multi_composite') == 'yes':
                    full_title += f" | Композит / Composite: Да ({metadata.get('composite_images_count','?')} сцени)"
                plt.title(full_title, fontsize=14, fontweight='bold', pad=20)
                plt.axis('off')
                plt.subplots_adjust(left=0.01, right=0.99, top=0.90, bottom=0.01)
                plt.show()
                print(f"✅ Показан RGB от / Displayed RGB from: {os.path.basename(image_path)}")
            else:
                print(f"❌ Не са намерени RGB канали / Could not find RGB bands. Налични: {band_descriptions}")
    except Exception as e:
        print(f"❌ Грешка при показване / Error displaying image: {e}")
        traceback.print_exc()

def create_square_bbox(lat: float, lon: float, size_km: float = SQUARE_SIZE_KM):
    """
    Създава квадратен BBOX около точката.
    Returns (min_lon, min_lat, max_lon, max_lat) for a square area.
    """
    R = 6371.0
    lat_offset = (size_km / 2) / R * (180 / math.pi)
    min_lat = lat - lat_offset
    max_lat = lat + lat_offset
    lon_offset = (size_km / 2) / (R * math.cos(math.radians(lat))) * (180 / math.pi)
    min_lon = lon - lon_offset
    max_lon = lon + lon_offset
    return (min_lon, min_lat, max_lon, max_lat)

def create_enhanced_nodata_mask(reflectance_bands):
    """
    Създава маска на липсващи/некачествени пиксели чрез рефлектанс канали.
    Creates a mask where pixels are likely no-data using reflectance bands.
    """
    if not reflectance_bands:
        return None
    bands_stack = np.array(reflectance_bands)
    zero_mask = np.all(bands_stack == 0, axis=0).astype(np.uint8)
    combined_mask = np.zeros_like(bands_stack[0], dtype=np.uint8)
    for band_data in reflectance_bands:
        suspicious = band_data < ENHANCED_NODATA_THRESHOLD
        combined_mask[suspicious] += 1
    suspicious_threshold = len(reflectance_bands) * 0.6
    nodata_mask = (combined_mask > suspicious_threshold).astype(np.uint8)
    enhanced_mask = np.logical_or(nodata_mask, zero_mask).astype(np.uint8)

    nodata_count = np.sum(enhanced_mask)
    total_pixels = enhanced_mask.size
    nodata_pct = (nodata_count / total_pixels) * 100
    zero_pct = (np.sum(zero_mask) / total_pixels) * 100
    suspicious_pct = (np.sum(nodata_mask) / total_pixels) * 100
    print(f"    Подобрена маска / Enhanced nodata mask: {nodata_pct:.1f}% ({nodata_count}/{total_pixels})")
    print(f"    Само нули / Zero-only: {zero_pct:.1f}%, Подозрителни / Suspicious: {suspicious_pct:.1f}%")
    return enhanced_mask

def get_reflectance_bands_only(all_bands_data, band_names):
    """
    Запазва само рефлектанс каналите (B01-B12, B8A).
    Keep only surface reflectance bands for robust nodata detection.
    """
    reflectance_bands = [b for b in band_names if (b.startswith('B') and len(b) == 3) or b == 'B8A']
    out_data = []
    out_names = []
    for bname, bdata in zip(band_names, all_bands_data):
        if bname in reflectance_bands:
            out_data.append(bdata)
            out_names.append(bname)
    return out_data, out_names

def get_transform_from_dataset(ds):
    """
    Извлича афинната трансформация от xarray Dataset.
    Extracts affine transform from an xarray Dataset.
    """
    try:
        if hasattr(ds, 'rio') and ds.rio.transform():
            transform = ds.rio.transform()
            print(f"    Трансформация от rio / Transform from rio: {transform}")
            return transform, abs(transform.a)
        x_res = float((ds.x[1] - ds.x[0]).values)
        y_res = float((ds.y[1] - ds.y[0]).values)
        transform = from_bounds(
            float(ds.x[0].values), float(ds.y[-1].values),
            float(ds.x[-1].values), float(ds.y[0].values),
            len(ds.x), len(ds.y)
        )
        print(f"    Ръчно изчислена трансформация / Manual transform: {transform}")
        return transform, abs(x_res)
    except Exception as e:
        print(f"    ⚠ Резервна трансформация / Transform fallback: {e}")
        return rasterio.Affine(10, 0, 0, 0, -10, 0), 10

def reproject_to_utm(bands_stack, src_transform, src_crs, target_epsg, bbox_geo):
    """
    Препроектира стек от изображения в целевата UTM проекция.
    Reprojects a band stack to the target UTM CRS.
    """
    dst_crs_obj = CRS.from_epsg(target_epsg)
    src_crs_obj = CRS.from_string(src_crs) if isinstance(src_crs, str) else src_crs
    resolution = abs(src_transform.a)

    transformer = Transformer.from_crs("EPSG:4326", dst_crs_obj, always_xy=True)
    min_lon, min_lat, max_lon, max_lat = bbox_geo
    corners = [(min_lon, min_lat), (min_lon, max_lat), (max_lon, min_lat), (max_lon, max_lat)]
    projected = [transformer.transform(lon, lat) for lon, lat in corners]
    xs = [p[0] for p in projected]
    ys = [p[1] for p in projected]
    dst_minx, dst_maxx = min(xs), max(xs)
    dst_miny, dst_maxy = min(ys), max(ys)

    dst_width = int(round((dst_maxx - dst_minx) / resolution))
    dst_height = int(round((dst_maxy - dst_miny) / resolution))
    dst_transform = from_origin(dst_minx, dst_maxy, resolution, -resolution)

    n_bands = bands_stack.shape[0]
    destination = np.zeros((n_bands, dst_height, dst_width), dtype=bands_stack.dtype)
    for i in range(n_bands):
        reproject(
            source=bands_stack[i],
            destination=destination[i],
            src_transform=src_transform,
            src_crs=src_crs_obj,
            dst_transform=dst_transform,
            dst_crs=dst_crs_obj,
            resampling=Resampling.bilinear
        )
    print(f"    Препроектирано → UTM {target_epsg}, форма / shape {destination.shape}")
    print(f"    Граници / Bounds: ({dst_minx:.1f},{dst_miny:.1f},{dst_maxx:.1f},{dst_maxy:.1f})")
    return destination, dst_transform

# ==================== ПОДДРЪЖКА НА МНОГОИЗОБРАЖЕН КОМПОЗИТ / MULTI‑IMAGE COMPOSITE SUPPORT ====================

def test_image_coverage(stac_item, fire_coords, nodata_mask):
    """
    Проверява колко от липсващите пиксели се покриват от тази сцена.
    Checks how many nodata pixels of the mask are covered by this item.
    """
    try:
        available_bands = list(stac_item.assets.keys())
        sentinel_bands = [b for b in available_bands if b.startswith('B') and b not in ['visual','preview']]
        if not sentinel_bands:
            return None
        cfg = yaml.safe_load("""
        sentinel-2-l2a:
          assets:
            "*":
              data_type: uint16
              nodata: 0
              unit: '1'
            "visual":
              data_type: uint8
              nodata: 0
              unit: '1'
        "*":
          warnings: ignore
        """)
        center_lat, center_lon = fire_coords
        bbox = create_square_bbox(center_lat, center_lon)
        test_band = 'B04' if 'B04' in sentinel_bands else sentinel_bands[0]
        ds = stac_load([stac_item], bands=[test_band], stac_cfg=cfg, bbox=bbox, chunks={})
        if ds.sizes['x'] == 0 or ds.sizes['y'] == 0:
            return None
        band_data = np.squeeze(ds[test_band].values)
        if band_data.shape != nodata_mask.shape:
            zoom_factors = (nodata_mask.shape[0] / band_data.shape[0],
                            nodata_mask.shape[1] / band_data.shape[1])
            band_data = zoom(band_data, zoom_factors, order=1)
        valid_data_mask = band_data > ENHANCED_NODATA_THRESHOLD / 2
        valid_in_nodata = np.sum((nodata_mask == 1) & valid_data_mask)
        total_nodata = np.sum(nodata_mask == 1)
        if total_nodata == 0:
            return {'coverage_percentage': 100}
        return {
            'coverage_percentage': (valid_in_nodata / total_nodata) * 100,
            'valid_pixels': valid_in_nodata,
            'total_nodata': total_nodata
        }
    except:
        return None

def find_complementary_images(catalog, central_datetime, fire_coords,
                              nodata_mask, search_days=COMPLEMENTARY_SEARCH_WINDOW_DAYS):
    """
    Търси допълнителни сцени в околността на основната дата.
    Searches for images within ±search_days of central_datetime
    that help fill the nodata_mask.
    """
    center_lat, center_lon = fire_coords
    start_date = central_datetime - timedelta(days=search_days)
    end_date = central_datetime + timedelta(days=search_days)
    datetime_range = f"{start_date.strftime('%Y-%m-%d')}/{end_date.strftime('%Y-%m-%d')}"
    print(f"  🔍 Търсене на допълнителни изображения / Searching complementary images: {datetime_range}")
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        datetime=datetime_range,
        intersects={"type": "Point", "coordinates": [center_lon, center_lat]},
        query={"eo:cloud_cover": {"lt": MAX_CLOUD_COVER}}
    )
    items = list(search.items())
    if not items:
        return []
    candidates = []
    for item in items[:20]:
        coverage_info = test_image_coverage(item, fire_coords, nodata_mask)
        if coverage_info and coverage_info['coverage_percentage'] >= MIN_COMPLEMENTARY_COVERAGE:
            candidates.append({
                'item': item,
                'coverage_percentage': coverage_info['coverage_percentage'],
                'cloud_cover': item.properties.get('eo:cloud_cover', 100),
                'date': item.datetime.replace(tzinfo=pytz.UTC)
            })
    candidates.sort(key=lambda x: (-x['coverage_percentage'], x['cloud_cover']))
    selected = [c['item'] for c in candidates[:MAX_COMPLEMENTARY_IMAGES]]
    return selected

def create_multi_image_composite(main_item, complementary_items, sentinel_bands,
                                 bbox, cfg, enhanced_nodata_mask, fire_coords):
    """
    Запълва липсващите пиксели с данни от допълнителни сцени.
    Fills nodata pixels in the main image using complementary images.
    """
    try:
        print(f"  🧩 Създаване на композит с {len(complementary_items)} доп. изображения / Creating composite")
        all_images_data = []
        all_image_info = []

        ds_main = stac_load([main_item], bands=sentinel_bands, stac_cfg=cfg, bbox=bbox, chunks={})
        main_bands = {}
        for band in sentinel_bands:
            if band in ds_main:
                data = np.nan_to_num(np.squeeze(ds_main[band].values), nan=0)
                main_bands[band] = data
        if not main_bands:
            return None
        all_images_data.append({
            'source': 'main', 'bands': main_bands,
            'cloud_cover': main_item.properties.get('eo:cloud_cover', 100),
            'date': main_item.datetime.replace(tzinfo=pytz.UTC)
        })
        all_image_info.append({
            'id': main_item.id, 'source': 'main',
            'cloud_cover': main_item.properties.get('eo:cloud_cover', 100),
            'date': main_item.datetime.strftime('%Y-%m-%d')
        })

        for i, comp_item in enumerate(complementary_items):
            ds_comp = stac_load([comp_item], bands=sentinel_bands, stac_cfg=cfg, bbox=bbox, chunks={})
            comp_bands = {}
            for band in sentinel_bands:
                if band in ds_comp:
                    data = np.nan_to_num(np.squeeze(ds_comp[band].values), nan=0)
                    ref_shape = list(main_bands.values())[0].shape
                    if data.shape != ref_shape:
                        zoom_factors = (ref_shape[0] / data.shape[0],
                                        ref_shape[1] / data.shape[1])
                        data = zoom(data, zoom_factors, order=1)
                    comp_bands[band] = data
            if comp_bands:
                all_images_data.append({
                    'source': f'comp_{i}', 'bands': comp_bands,
                    'cloud_cover': comp_item.properties.get('eo:cloud_cover', 100),
                    'date': comp_item.datetime.replace(tzinfo=pytz.UTC)
                })
                all_image_info.append({
                    'id': comp_item.id, 'source': f'comp_{i}',
                    'cloud_cover': comp_item.properties.get('eo:cloud_cover', 100),
                    'date': comp_item.datetime.strftime('%Y-%m-%d')
                })

        composite_bands = {}
        band_usage_stats = {}
        for band in sentinel_bands:
            if band in main_bands:
                composite = main_bands[band].copy()
                band_stack = [img['bands'][band] for img in all_images_data if band in img['bands']]
                nodata_positions = np.where(enhanced_nodata_mask == 1)
                total_nodata = len(nodata_positions[0])
                filled_count = 0
                for idx in range(total_nodata):
                    y, x = nodata_positions[0][idx], nodata_positions[1][idx]
                    if band_stack[0][y, x] > ENHANCED_NODATA_THRESHOLD / 2:
                        composite[y, x] = band_stack[0][y, x]
                        filled_count += 1
                        continue
                    for comp_idx in range(1, len(band_stack)):
                        if band_stack[comp_idx][y, x] > ENHANCED_NODATA_THRESHOLD / 2:
                            composite[y, x] = band_stack[comp_idx][y, x]
                            filled_count += 1
                            break
                fill_percentage = (filled_count / total_nodata) * 100 if total_nodata else 100
                composite_bands[band] = composite
                band_usage_stats[band] = {'filled': fill_percentage, 'total_nodata': total_nodata}
        total_filled = sum(s['filled'] * s['total_nodata'] for s in band_usage_stats.values())
        total_nodata_all = sum(s['total_nodata'] for s in band_usage_stats.values())
        overall_fill = (total_filled / total_nodata_all * 100) if total_nodata_all else 100
        print(f"     Общо запълване / Overall fill: {overall_fill:.1f}%")
        return {
            'composite_bands': composite_bands,
            'band_usage_stats': band_usage_stats,
            'image_info': all_image_info,
            'overall_fill_percentage': overall_fill
        }
    except Exception as e:
        print(f"Грешка при композит / Error creating composite: {e}")
        return None

# ==================== ОСНОВНА ЛОГИКА ЗА ИЗТЕГЛЯНЕ / MAIN DOWNLOAD LOGIC ====================

def download_monthly_square(item, fire_id, fire_coords, month_str, year, target_utm_epsg):
    """
    Изтегля и записва квадрат 10×10 км за даден месец.
    Downloads and saves a 10×10 km square for a given month.
    """
    center_lat, center_lon = fire_coords
    bbox = create_square_bbox(center_lat, center_lon)
    min_lon, min_lat, max_lon, max_lat = bbox

    # Име на файла / File name
    acq_date_str = item.datetime.strftime('%Y%m%d')
    out_filename = f"square_10km_allbands_{year}_{month_str}_{fire_id}_{acq_date_str}.tif"
    out_path = os.path.join(OUTPUT_DIR, out_filename)

    if os.path.exists(out_path):
        try:
            with rasterio.open(out_path) as src:
                _ = src.count
            print(f"  📁 Вече съществува / Already exists: {out_filename}")
            display_rgb_from_all_bands(out_path, f"{month_str.capitalize()} {fire_id} | СЪЩЕСТВУВА")
            return True
        except:
            print("  ⚠ Повреден файл, преизтегляне / Corrupt file, re‑downloading...")
            os.remove(out_path)

    print(f"  📥 Изтегляне / Downloading: {out_filename}")

    cfg = yaml.safe_load("""
    sentinel-2-l2a:
      assets:
        "*":
          data_type: uint16
          nodata: 0
          unit: '1'
        "visual":
          data_type: uint8
          nodata: 0
          unit: '1'
        "AOT":
          data_type: uint16
          nodata: 0
          unit: '1'
        "WVP":
          data_type: uint16
          nodata: 0
          unit: '1'
        "SCL":
          data_type: uint8
          nodata: 0
          unit: '1'
    "*":
      warnings: ignore
    """)

    available_bands = list(item.assets.keys())
    sentinel_bands = [b for b in available_bands
                      if b.startswith(('B', 'AOT', 'SCL', 'WVP'))
                      and b not in ['visual', 'preview']]
    sentinel_bands.sort()
    print(f"  Извличане на канали / Extracting bands: {sentinel_bands}")

    ds = stac_load([item], bands=sentinel_bands, stac_cfg=cfg,
                   bbox=[min_lon, min_lat, max_lon, max_lat], chunks={})
    if ds.sizes['x'] == 0 or ds.sizes['y'] == 0:
        print("  ✗ Няма данни за този BBOX / No data returned for this bbox")
        return False

    # Определяне на CRS / CRS detection
    if hasattr(ds, 'rio') and ds.rio.crs is not None:
        source_crs_obj = ds.rio.crs
    elif 'crs' in ds.attrs and ds.attrs['crs'] is not None:
        crs_val = ds.attrs['crs']
        source_crs_obj = CRS.from_string(crs_val) if isinstance(crs_val, str) else crs_val
    else:
        x_val = float(ds.x[0].values)
        if x_val > 1000:
            source_crs_obj = CRS.from_epsg(target_utm_epsg)
        else:
            source_crs_obj = CRS.from_epsg(4326)

    source_crs_epsg = source_crs_obj.to_epsg()
    print(f"  Изходен CRS / Source CRS: EPSG:{source_crs_epsg}")

    all_bands_data = []
    band_names = []
    for band in sentinel_bands:
        if band in ds:
            data = np.nan_to_num(np.squeeze(ds[band].values), nan=0)
            if np.max(data) == 0:
                print(f"  ⚠ Канал {band} е изцяло нулев / entirely zero – пропуска се")
                continue
            all_bands_data.append(data)
            band_names.append(band)
        else:
            print(f"  ⚠ Канал {band} не е намерен / not found")

    if not all_bands_data:
        print("  ✗ Няма валидни канали / No valid bands loaded")
        return False

    refl_bands, refl_names = get_reflectance_bands_only(all_bands_data, band_names)
    if not refl_bands:
        print("  ✗ Няма рефлектанс канали / No reflectance bands – не може да се оцени nodata")
        return False
    enhanced_nodata_mask = create_enhanced_nodata_mask(refl_bands)
    nodata_percentage = (np.sum(enhanced_nodata_mask) / enhanced_nodata_mask.size) * 100
    print(f"  Начален % nodata (рефлектанс) / Initial nodata: {nodata_percentage:.1f}%")

    is_multi_composite = False
    composite_result = None

    if nodata_percentage > MAX_NODATA_PERCENTAGE:
        catalog = pystac_client.Client.open(
            'https://planetarycomputer.microsoft.com/api/stac/v1',
            modifier=planetary_computer.sign_inplace
        )
        central_dt = item.datetime.replace(tzinfo=pytz.UTC)
        complementary_items = find_complementary_images(
            catalog, central_dt, fire_coords, enhanced_nodata_mask
        )
        if complementary_items:
            composite_result = create_multi_image_composite(
                item, complementary_items, sentinel_bands,
                [min_lon, min_lat, max_lon, max_lat], cfg,
                enhanced_nodata_mask, fire_coords
            )
            if composite_result and composite_result['composite_bands']:
                is_multi_composite = True

    if is_multi_composite:
        final_bands = []
        final_names = []
        for band in sentinel_bands:
            if band in composite_result['composite_bands']:
                final_bands.append(composite_result['composite_bands'][band])
                final_names.append(band)
        overall_fill = composite_result['overall_fill_percentage']
    else:
        final_bands = all_bands_data
        final_names = band_names
        overall_fill = 100.0

    # Подравняване на размерите / Align shapes
    shapes = [b.shape for b in final_bands]
    if len(set(shapes)) > 1:
        target_shape = max(set(shapes), key=shapes.count)
        for i, b in enumerate(final_bands):
            if b.shape != target_shape:
                zoom_factors = (target_shape[0] / b.shape[0],
                                target_shape[1] / b.shape[1])
                final_bands[i] = zoom(b, zoom_factors, order=1)

    bands_stack = np.stack(final_bands, axis=0)
    print(f"  Форма на стека / Stack shape: {bands_stack.shape}")

    transform_src, resolution_m = get_transform_from_dataset(ds)

    if source_crs_epsg != target_utm_epsg:
        print(f"  🔄 Препроектиране / Reprojecting EPSG:{source_crs_epsg} → EPSG:{target_utm_epsg}")
        bands_stack, transform_out = reproject_to_utm(
            bands_stack, transform_src, source_crs_obj, target_utm_epsg, bbox
        )
        if bands_stack is None or np.max(bands_stack) == 0:
            print("  ❌ Препроекцията даде нулев резултат / Reprojection produced empty data")
            return False
    else:
        transform_out = transform_src
        print(f"  ✅ Вече в целевия UTM / Already in target UTM")

    if np.max(bands_stack) == 0:
        print("  ❌ Крайният стек е изцяло нулев / Final stack is entirely zero")
        return False

    # Финална оценка на nodata / Final nodata
    final_refl = []
    for i, bname in enumerate(final_names):
        if bname in refl_names:
            final_refl.append(bands_stack[i])
    if final_refl:
        final_mask = create_enhanced_nodata_mask(final_refl)
        final_nodata_pct = (np.sum(final_mask) / final_mask.size) * 100 if final_mask is not None else 0
    else:
        final_nodata_pct = (np.sum(bands_stack[0] == 0) / bands_stack[0].size) * 100

    profile = {
        'driver': 'GTiff',
        'height': bands_stack.shape[1],
        'width': bands_stack.shape[2],
        'count': bands_stack.shape[0],
        'dtype': bands_stack.dtype,
        'crs': CRS.from_epsg(target_utm_epsg).to_wkt(),
        'transform': transform_out,
        'compress': 'deflate',
        'nodata': 0,
    }

    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write(bands_stack)
        for i, name in enumerate(final_names, 1):
            dst.set_band_description(i, name)

        tags = {
            'fire_id': fire_id,
            'fire_lat': center_lat, 'fire_lon': center_lon,
            'month': month_str,
            'year': str(year),
            'acquisition_date': item.datetime.strftime('%Y-%m-%d'),
            'cloud_cover': item.properties.get('eo:cloud_cover', 'unknown'),
            'initial_nodata_percentage': f"{nodata_percentage:.1f}",
            'final_nodata_percentage': f"{final_nodata_pct:.1f}",
            'is_multi_composite': "yes" if is_multi_composite else "no",
            'target_utm_epsg': str(target_utm_epsg)
        }
        if is_multi_composite and composite_result:
            tags['composite_type'] = 'multi_image'
            tags['composite_images_count'] = str(len(composite_result['image_info']))
            tags['composite_overall_fill'] = f"{overall_fill:.1f}"
            for i, info in enumerate(composite_result['image_info']):
                prefix = 'main' if i == 0 else f'comp_{i}'
                tags[f'{prefix}_image_id'] = info['id']
                tags[f'{prefix}_date'] = info['date']
                tags[f'{prefix}_cloud_cover'] = str(info['cloud_cover'])
        dst.update_tags(**tags)

    file_size_mb = os.path.getsize(out_path) / (1024**2)
    print(f"  ✅ Записан / Saved: {out_filename} ({file_size_mb:.1f} MB)")
    print(f"     UTM EPSG:{target_utm_epsg}, форма {bands_stack.shape}, финален nodata: {final_nodata_pct:.1f}%")

    display_rgb_from_all_bands(out_path, f"{month_str.capitalize()} Пожар {fire_id} | НОВ")
    return True

# ==================== ГЛАВНА ПРОГРАМА / MAIN ====================

def main():
    print("Изтегляне на 10×10 км месечни Sentinel‑2 квадрати за 2024 г.")
    print("Downloading 10×10 km monthly Sentinel‑2 squares for 2024")
    print("=" * 70)
    print(f"Макс. облачност / Max cloud: {MAX_CLOUD_COVER}%")
    print(f"Композит при nodata > {MAX_NODATA_PERCENTAGE}%")
    print(f"Входен CSV / Input: {INPUT_CSV}")
    print(f"Изходна папка / Output: {OUTPUT_DIR}")
    print("=" * 70)

    # Зареждане на пожари / Load fires
    df = pd.read_csv(INPUT_CSV)
    required = ['Fire_id', 'Lat', 'Lon']
    missing = [c for c in required if c not in df.columns]
    if missing:
        print(f"❌ Липсващи колони / Missing columns: {missing}")
        return
    fires = df[['Fire_id', 'Lat', 'Lon']].dropna()
    print(f"Заредени {len(fires)} пожара от CSV / Loaded {len(fires)} fires.")

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Свързване с каталога / Connect to STAC
    catalog = pystac_client.Client.open(
        'https://planetarycomputer.microsoft.com/api/stac/v1',
        modifier=planetary_computer.sign_inplace
    )

    months = {
        5: 'may',
        7: 'july',
        9: 'september'
    }
    year = 2024

    for _, row in fires.iterrows():
        fire_id = str(int(row['Fire_id']))
        lat = float(row['Lat'])
        lon = float(row['Lon'])
        fire_coords = (lat, lon)
        target_epsg = get_utm_epsg(lon)
        print(f"\n{'='*50}\nПожар / Fire {fire_id} (EPSG:{target_epsg})\n{'='*50}")

        for month_num, month_name in months.items():
            print(f"\n--- Обработка на / Processing {month_name.capitalize()} {year} ---")

            # Първично търсене в месеца
            first_day = datetime(year, month_num, 1)
            last_day = datetime(year, month_num, calendar.monthrange(year, month_num)[1])
            datetime_range = f"{first_day.strftime('%Y-%m-%d')}/{last_day.strftime('%Y-%m-%d')}"
            print(f"  Първично търсене / Primary search: {datetime_range}")

            search = catalog.search(
                collections=["sentinel-2-l2a"],
                datetime=datetime_range,
                intersects={"type": "Point", "coordinates": [lon, lat]},
                query={"eo:cloud_cover": {"lt": MAX_CLOUD_COVER}}
            )
            items = list(search.items())

            # Ако няма – търси следващия месец
            if not items:
                next_month = month_num + 1
                if next_month > 12:
                    print(f"  ❌ Няма следващ месец за {month_name}, пропуска се / No next month, skipping.")
                    continue
                first_day_next = datetime(year, next_month, 1)
                last_day_next = datetime(year, next_month, calendar.monthrange(year, next_month)[1])
                datetime_range = f"{first_day_next.strftime('%Y-%m-%d')}/{last_day_next.strftime('%Y-%m-%d')}"
                print(f"  Няма в основния месец, разширяване към / Expanding to: {datetime_range}")
                search = catalog.search(
                    collections=["sentinel-2-l2a"],
                    datetime=datetime_range,
                    intersects={"type": "Point", "coordinates": [lon, lat]},
                    query={"eo:cloud_cover": {"lt": MAX_CLOUD_COVER}}
                )
                items = list(search.items())
                if not items:
                    print(f"  ❌ Все още няма подходящи изображения за {month_name}, пропуска се / Still none, skipping.")
                    continue

            # Избор на най-доброто изображение (най-ниска облачност)
            items.sort(key=lambda x: x.properties.get('eo:cloud_cover', 100))
            best_item = items[0]
            cc = best_item.properties.get('eo:cloud_cover', '?')
            print(f"  Избрано изображение / Selected: {best_item.id} (облачност / cloud: {cc}%)")

            # Изтегляне и запис
            success = download_monthly_square(
                best_item, fire_id, fire_coords, month_name, year, target_epsg
            )
            if success:
                print(f"  ✅ Квадратът за {month_name.capitalize()} е записан / square saved.")

    print("\n" + "=" * 70)
    print("ОБРАБОТКАТА ЗАВЪРШИ / PROCESSING COMPLETED.")
    square_files = [f for f in os.listdir(OUTPUT_DIR) if f.startswith('square_10km_allbands_')]
    print(f"Общ брой файлове / Total files: {len(square_files)}")

if __name__ == "__main__":
    main()